# Tag a split on Kaggle

This notebook runs one CARIS step, `caris tag`, on a free Kaggle GPU. Everything else runs on your own machine.

Before running it:

- **Settings:** accelerator *GPU T4 x2*, internet *on*.
- **Input:** a Kaggle dataset holding the `splits/` and `images/` folders that `caris split` wrote into `data/`. Leave the dataset zip out.
- **Code:** set `REPO` below to a URL that Kaggle can clone. For a private repository, put a GitHub token in the URL from a Kaggle secret, or upload the repository as a second dataset and install it from there.

Run it with **Save Version → Save & Run All**, so it carries on with the browser closed.

Tagging stops `TAGGING_HOURS` after the start, so the version ends, and keeps its output, before Kaggle ends the session. If a tag run isn't done by then, the last code cell says so. To carry on:

1. Add the output of the version that stopped to this notebook as an input.
2. Set `PREVIOUS` below to the folder that input is mounted at.
3. Save and run a new version. It copies the tag runs from `PREVIOUS` and tags only the images they don't have yet.

In [ ]:
REPO = "https://github.com/<you>/CARIS_v2.git"
DATA = "/kaggle/input/caris-data"  # the dataset with splits/ and images/
EXPERIMENT = "experiments/gemma4-26b__mxbai__standalone.toml"
SPLITS = ["val", "test"]
OLLAMA_VERSION = "0.40.16"  # the version you run locally, so both machines tag alike
TAGGING_HOURS = 8.5  # stop tagging this long after the start; keep it below your session limit
PREVIOUS = None  # to carry on: where an earlier version's output is mounted, e.g. "/kaggle/input/<this-notebook>"

In [ ]:
import subprocess
import time
import urllib.request

STARTED = time.time()  # TAGGING_HOURS counts from here

!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):  # wait until the server answers
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version")
        break
    except OSError:
        time.sleep(1)

In [ ]:
!git clone --depth 1 {REPO} /kaggle/working/caris
!pip install --quiet -e /kaggle/working/caris
%cd /kaggle/working/caris

In [ ]:
import shutil
from pathlib import Path

# Carry on from an earlier version: take each of its tag runs that got further than the copy here.
runs = Path("runs/tags")
runs.mkdir(parents=True, exist_ok=True)
if PREVIOUS:
    earlier_runs = sorted(Path(PREVIOUS).glob("**/runs/tags/*.jsonl"))
    if not earlier_runs:
        print(f"No tag runs under {PREVIOUS}. Check the path; this version starts from the runs in the repository.")
    for earlier in earlier_runs:
        here = runs / earlier.name
        if not here.exists() or here.stat().st_size < earlier.stat().st_size:
            shutil.copyfile(earlier, here)
            print(f"Carrying on from {earlier}")

In [ ]:
import tomllib

with open(EXPERIMENT, "rb") as f:
    model = tomllib.load(f)["tagging"]["model"]
!ollama pull {model}

In [ ]:
import os

os.environ["CARIS_DATA"] = DATA
for split in SPLITS:
    seconds_left = int(STARTED + TAGGING_HOURS * 3600 - time.time())
    if seconds_left <= 0:
        print(f"No time left to tag {split}.")
        break
    !timeout {seconds_left} caris tag {EXPERIMENT} --split {split}
    if _exit_code == 124:  # timeout stopped it; every image tagged so far is in the run
        print(f"Out of time while tagging {split}.")
        break
    if _exit_code:
        print(f"caris tag failed on {split}: see its message above.")
        break

In [ ]:
import json

# How far each tag run has got. Failed images count as done: their line records the failure.
for run in sorted(Path("runs/tags").glob("*.jsonl")):
    with run.open(encoding="utf-8") as f:
        split = json.loads(next(f))["header"]["split"]
        images = sum(1 for line in f if line.endswith("\n"))
    total = len(json.loads(Path(DATA, "splits", f"{split}.json").read_text(encoding="utf-8")))
    status = "done" if images >= total else "not done: carry on in a new version (see the top of the notebook)"
    print(f"{run.name}: {images} of {total} images, {status}")

The tag runs are in `/kaggle/working/caris/runs/tags/`. Once the cell above says they are done, download them from the notebook's **Output** tab, put them in `runs/tags/` of your local repository, and commit them.